# B-spline KAN experiments (KAN-1, KAN-2, KAN-4; KAN-3 if time)
**Owner:** Abdelrahman Elbehery · Plan steps 6–10

| Section | Output on Drive |
|---|---|
| 1. Timing pilot | numbers to agree the shared training budget |
| 2. Accuracy runs (Adam, shared `train()`) | `results/bkan.csv` |
| 3. Symbolic pipeline (LBFGS, prune, snap) | `results/bkan_recovery.csv` |
| 4. Edge-function figure | `figures/bkan_edges_<eq>.pdf` |
| 5. Quick look at results | — |

**Needs from teammates:** `src/data.py` (`load_problem`, `list_problems`), `src/metrics.py` (`regression_metrics`).
**Needs from you:** `src/models/bspline_kan.py` (plan step 5).
Every loop saves after each run and skips finished runs, so after a disconnect: rerun the setup cell and the section you were in.

## 0. Setup

In [ ]:
# ---- Colab setup: run this cell first in every session ----
import os, subprocess
from google.colab import drive, userdata
drive.mount("/content/drive")

REPO_URL = "https://github.com/Elbehery-237/ann-group13.git"   # change if the repo moves
REPO = "/content/ann-group13"

if os.path.exists(REPO):
    !git -C {REPO} pull -q                                   # new session in same VM: just update
else:
    try:                                                     # private repo: token stored in Colab Secrets
        token = userdata.get("GITHUB_TOKEN")
        url = REPO_URL.replace("https://", f"https://{token}@")
    except Exception:                                        # public repo: no token needed
        url = REPO_URL
    !git clone -q {url} {REPO}
%cd {REPO}
!pip install -q -r requirements.txt

# Shared paths on Google Drive (data downloaded once, results survive disconnects)
DRIVE = "/content/drive/MyDrive/ann-group13"
os.environ["SRSD_ROOT"] = f"{DRIVE}/data/srsd-feynman_medium"
RESULTS, FIGURES = f"{DRIVE}/results", f"{DRIVE}/figures"
os.makedirs(RESULTS, exist_ok=True); os.makedirs(FIGURES, exist_ok=True)

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "| data found:", os.path.exists(os.environ["SRSD_ROOT"]))

In [ ]:
import os, time
import numpy as np, pandas as pd, sympy as sp, torch
import matplotlib.pyplot as plt

from src.data import load_problem, list_problems              # S1 (Jomana)
from src.metrics import regression_metrics                    # S3 (A. Mohamed)
from src.train import DEFAULTS, set_seed, train               # S2 (you)
from src.symbolic import load_true_eq, to_original, is_exact, ned   # S5 (you)
from src.models.bspline_kan import BSplineKAN, grid_update_hook     # your model

SEEDS = (0, 1, 2)
EQUATIONS = list_problems()          # all 40 Medium problems (or the agreed fixed subset)
print(len(EQUATIONS), "equations | budget:", DEFAULTS)


def evaluate_test(model, data):
    """Test metrics in ORIGINAL units (undo target standardization), plus inference time."""
    model.eval()
    X = data["Xte"].to(DEVICE)
    with torch.no_grad():
        t0 = time.perf_counter(); y_scaled = model(X); infer_ms = 1000 * (time.perf_counter() - t0)
    ys = data["yscaler"]
    y_pred = y_scaled.cpu().double().numpy().ravel() * ys.std + ys.mean
    y_true = data["yte"].cpu().double().numpy().ravel() * ys.std + ys.mean
    return dict(infer_time_ms=infer_ms, **regression_metrics(y_true, y_pred))


def append_row(path, row):
    """Append one result row to a CSV on Drive (header written only once)."""
    pd.DataFrame([row]).to_csv(path, mode="a", header=not os.path.exists(path), index=False)


def finished(path, keys=("spec", "equation", "seed")):
    if not os.path.exists(path):
        return set()
    return set(pd.read_csv(path)[list(keys)].itertuples(index=False, name=None))

## 1. Timing pilot (plan step 6)
KAN-1 for 200 epochs on three equations with 1, 3 and 6 inputs, on GPU and on CPU.
Use the seconds per epoch to choose `max_epochs` and `patience` **with the team**, then set them in `src/train.py` → `DEFAULTS` and push.

In [ ]:
PILOT = ["feynman-i.29.4", "feynman-i.12.2", "feynman-i.11.19"]   # d = 1, 3, 6
rows = []
for dev in (["cuda", "cpu"] if torch.cuda.is_available() else ["cpu"]):
    for eq in PILOT:
        set_seed(0); data = load_problem(eq)
        m = BSplineKAN(data["d"], hidden=(5,), grid=5, k=3, seed=0, device=dev)
        _, _, info = train(m, data, max_epochs=200, patience=10**9, device=dev)
        rows.append(dict(device=dev, equation=eq, d=data["d"], s_per_epoch=info["time"] / info["epochs"]))
pilot = pd.DataFrame(rows); display(pilot)

# Rough total for your runs: 40 eq x 3 seeds x (KAN-1 + KAN-4 + 3 KAN-2 stages) x epochs x s/epoch
best = pilot.groupby("device").s_per_epoch.mean().min()
for epochs in (500, 1000, 2000, 5000):
    hours = 40 * 3 * 5 * epochs * best / 3600
    print(f"max ~{epochs:>5} epochs/run -> about {hours:5.1f} h (upper bound; early stopping shortens it)")

## 2. Accuracy runs (plan step 7)
All accuracy numbers come from the shared Adam harness. Order: KAN-1 → KAN-4 → KAN-2 (→ KAN-3 if time).
KAN-2 saves one row per grid stage (`KAN-2/G=5`, `KAN-2/G=10`, `KAN-2/G=20`) for the staircase curve.
Diverged runs stay in the file with `diverged=True`.

In [ ]:
OUT = f"{RESULTS}/bkan.csv"
SPECS = {                                     # spec -> constructor hyperparameters (suggested)
    "KAN-1": dict(hidden=(5,), grid=5, k=3),
    "KAN-4": dict(hidden=([5, 2],), grid=5, k=3),
    # "KAN-3": dict(hidden=(5,), grid=5, k=2),   # only if time
}
KAN2_GRIDS = (5, 10, 20)


def result_row(spec, eq, seed, model, data, info):
    return dict(model="bkan", spec=spec, equation=eq, seed=seed,
                n_params=model.count_params(), train_time_s=info["time"],
                epochs_run=info["epochs"], diverged=info["diverged"],
                val_mse_scaled=info["best_val"], **evaluate_test(model, data))


done = finished(OUT)
for spec, hp in SPECS.items():
    for eq in EQUATIONS:
        for seed in SEEDS:
            if (spec, eq, seed) in done:
                continue
            set_seed(seed); data = load_problem(eq)
            model = BSplineKAN(data["d"], **hp, seed=seed, device=DEVICE)
            model, hist, info = train(model, data, hooks=(grid_update_hook,), device=DEVICE)
            append_row(OUT, result_row(spec, eq, seed, model, data, info))
            print(f"{spec} {eq} seed={seed}: epochs={info['epochs']} diverged={info['diverged']}")

In [ ]:
# KAN-2: grid extension 5 -> 10 -> 20. refine() creates new parameters, so train() (fresh Adam) runs per stage.
done = finished(OUT)
for eq in EQUATIONS:
    for seed in SEEDS:
        if (f"KAN-2/G={KAN2_GRIDS[-1]}", eq, seed) in done:
            continue
        set_seed(seed); data = load_problem(eq)
        model = BSplineKAN(data["d"], hidden=(5,), grid=KAN2_GRIDS[0], k=3, seed=seed, device=DEVICE)
        for s, G in enumerate(KAN2_GRIDS):
            if s > 0:
                model.kan = model.kan.refine(G)
            model, hist, info = train(model, data, hooks=(grid_update_hook,) if s == 0 else (), device=DEVICE)
            if (f"KAN-2/G={G}", eq, seed) not in done:
                append_row(OUT, result_row(f"KAN-2/G={G}", eq, seed, model, data, info))
            if info["diverged"]:
                break
        print(f"KAN-2 {eq} seed={seed} done")

## 3. Symbolic pipeline (plan steps 8–9)
A separate run from the accuracy runs: pykan's LBFGS `fit()` with a sparsity penalty, then prune, retrain, snap edges to symbolic functions, refit constants, convert to original variables and score.
Can run on a **CPU runtime** in a second Colab tab while section 2 runs on the GPU.
Try `EQUATIONS_SYM = PILOT_SYM` first, check the formulas look sensible, then run all.

In [ ]:
OUT_SYM = f"{RESULTS}/bkan_recovery.csv"
LAMBDAS = (1e-4, 1e-3, 1e-2)                                  # suggested; chosen on validation
LIB = ["x", "x^2", "x^3", "x^4", "sqrt", "1/x", "exp", "log", "sin", "cos"]
ROUND_DIGITS = 4                                              # report this value in the paper
NOLOG = 10**9                                                 # pykan 0.2.8: fit(log=0) crashes
PILOT_SYM = ["feynman-i.39.10", "feynman-i.11.19", "feynman-iii.17.37"]
EQUATIONS_SYM = PILOT_SYM                                     # then: EQUATIONS


def nrmse_scaled(m, X, y):                                    # model selection only
    with torch.no_grad():
        return (torch.sqrt(torch.mean((m(X) - y) ** 2)) / y.std()).item()


done_sym = finished(OUT_SYM, keys=("equation",))
for eq in EQUATIONS_SYM:
    if (eq,) in done_sym:
        continue
    data = load_problem(eq)
    ds = dict(train_input=data["Xtr"], train_label=data["ytr"],
              test_input=data["Xva"], test_label=data["yva"])     # pykan "test" = our validation
    best = None
    for lamb in LAMBDAS:
        set_seed(0)
        m = BSplineKAN(data["d"], hidden=(5,), grid=5, k=3, seed=0)
        m.kan.fit(ds, opt="LBFGS", steps=200, lamb=lamb, log=NOLOG)
        va = nrmse_scaled(m, data["Xva"], data["yva"])
        if best is None or va < best[0]:
            best = (va, lamb, m)
    _, lamb, m = best
    unpruned = nrmse_scaled(m, data["Xte"], data["yte"])
    m.kan = m.kan.prune(); m.kan.fit(ds, opt="LBFGS", steps=100, log=NOLOG)
    pruned = nrmse_scaled(m, data["Xte"], data["yte"])
    m.kan.auto_symbolic(lib=LIB, verbose=0)
    m.kan.fit(ds, opt="LBFGS", steps=50, update_grid=False, log=NOLOG)
    expr = to_original(m.formula(), data["xscaler"], data["yscaler"], data["d"])
    true = load_true_eq(eq, os.environ["SRSD_ROOT"])
    row = dict(equation=eq, lamb=lamb, recovered_formula=str(sp.N(expr, 5)),
               exact=is_exact(expr, true, round_digits=ROUND_DIGITS), ned=ned(expr, true),
               test_nrmse_unpruned=unpruned, test_nrmse_pruned=pruned, true_formula=str(true))
    append_row(OUT_SYM, row)
    print(eq, "| exact:", row["exact"], "| NED:", round(row["ned"], 3), "|", row["recovered_formula"])

## 4. Edge-function figure (plan step 10)
For one **recovered** equation (ideally a product law), plot the pruned KAN with the symbolic function each edge was snapped to.
Re-runs the pipeline for that one equation so the model is in memory.

In [ ]:
FIG_EQ = "feynman-i.39.10"                                    # pick one with exact=True
data = load_problem(FIG_EQ)
ds = dict(train_input=data["Xtr"], train_label=data["ytr"], test_input=data["Xva"], test_label=data["yva"])
lamb = pd.read_csv(OUT_SYM).set_index("equation").loc[FIG_EQ, "lamb"]
set_seed(0)
m = BSplineKAN(data["d"], hidden=(5,), grid=5, k=3, seed=0)
m.kan.fit(ds, opt="LBFGS", steps=200, lamb=lamb, log=NOLOG)
m.kan = m.kan.prune(); m.kan.fit(ds, opt="LBFGS", steps=100, log=NOLOG)
m.kan.auto_symbolic(lib=LIB, verbose=0)
m.kan(data["Xtr"])                                            # store activations for plotting
m.kan.plot(in_vars=[f"x{i}" for i in range(data["d"])], out_vars=["y"], folder="/content/kan_plot_tmp")
plt.savefig(f"{FIGURES}/bkan_edges_{FIG_EQ}.pdf", bbox_inches="tight")
plt.show()

## 5. Quick look at results (final tables come from Yousef's `aggregate.py`)

In [ ]:
res = pd.read_csv(f"{RESULTS}/bkan.csv")
summary = (res.groupby("spec")
             .agg(runs=("seed", "size"), diverged=("diverged", "sum"),
                  median_nrmse=("nrmse", "median"), median_r2=("r2", "median"),
                  median_params=("n_params", "median")))
display(summary)

if os.path.exists(f"{RESULTS}/bkan_recovery.csv"):
    rec = pd.read_csv(f"{RESULTS}/bkan_recovery.csv")
    print(f"exact recoveries: {rec.exact.sum()} / {len(rec)} | median NED: {rec.ned.median():.3f}")
    display(rec[["equation", "exact", "ned", "recovered_formula", "true_formula"]])